# 🌿 SAMSUNG INNOVATION CAMPUS (SIC) - CAPSTONE PROJECT
## Đề tài: Hệ thống tư vấn hỗ trợ bệnh trên cây trồng có tích hợp mô hình AI phân tích bệnh qua hình ảnh
*(AI-Integrated Crop Disease Advisory and Image-Based Diagnosis System)*

* **Nhóm thực hiện:** Group 5
* **Phân công trách nhiệm:**
  * **Nguyễn Quốc Khánh (Backend Developer & Database Architect)**: Xây dựng RESTful API (FastAPI), tích hợp inference mô hình AI, thiết kế cơ sở dữ liệu chuyên gia nông nghiệp (Advisory Knowledge Base).
  * **Đinh Gia Bảo (Team Leader & ML Engineer)**: Xây dựng & huấn luyện mạng CNN MobileNetV2.
  * **Trần Đặng Công Tâm (Data & Optimization Engineer)**: Thu thập, tiền xử lý & tối ưu hóa tập dữ liệu PlantVillage.
  * **Ma Hồng Hải (Frontend Developer & Agronomic QA)**: Xây dựng giao diện web responsive, kiểm định kiến thức nông học.

---
### 🎯 Mục tiêu của Notebook này:
Notebook này là môi trường phát triển và kiểm thử toàn diện của Backend, bao gồm:
1. **Module 1:** Thiết kế & truy vấn Cơ sở dữ liệu Tri thức Nông nghiệp (Knowledge Base).
2. **Module 2:** Xây dựng Pipeline xử lý ảnh & Input Validation (kích thước, định dạng, lọc file lỗi).
3. **Module 3:** Tích hợp mô hình MobileNetV2 Inference Engine & Đo lường thời gian đáp ứng (Latency Benchmark < 200ms).
4. **Module 4:** Bộ điều phối logic quyết định (Confidence-Gated Decision Engine) với ngưỡng tin cậy an toàn.
5. **Module 5:** Xây dựng và kiểm thử RESTful API chuẩn OpenAPI với FastAPI.
6. **Module 6:** Chạy tự động trọn bộ End-to-End Test Suite.


In [ ]:
import os
import sys
import io
import time
import json
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Thêm thư mục gốc vào PYTHONPATH để nạp các module backend
current_dir = Path(".").resolve()
if str(current_dir) not in sys.path:
    sys.path.insert(0, str(current_dir))

import torch
import torchvision
from fastapi.testclient import TestClient

from backend.config import (
    KNOWLEDGE_BASE_PATH,
    CONFIDENCE_THRESHOLD,
    CLASS_NAMES,
    MODEL_INPUT_SIZE,
    IMAGENET_MEAN,
    IMAGENET_STD,
    SAMPLE_IMAGES_DIR
)
from backend.inference_engine import (
    CropDiseaseInferenceEngine,
    validate_image_file,
    preprocess_image,
    InputValidationError
)
from backend.advisory_engine import AdvisoryKnowledgeBase
from backend.main import app

print("[OK] Đã nạp thành công các thư viện & module Backend!")
print(f"PyTorch: {torch.__version__} | TorchVision: {torchvision.__version__}")
print(f"Số lớp bệnh hỗ trợ: {len(CLASS_NAMES)}")
print(f"Ngưỡng tin cậy an toàn (Confidence Threshold): {CONFIDENCE_THRESHOLD * 100:.0f}%")


---
## 🌾 Module 1: Cơ sở Dữ liệu Tri thức Nông nghiệp (Subsystem B - Knowledge Base)

Theo yêu cầu chuẩn của FAO, USDA và Viện Bảo vệ Thực vật Việt Nam, cơ sở dữ liệu tri thức được lưu trữ dưới dạng cấu trúc JSON chi tiết, ánh xạ từng loại bệnh với:
* Tên khoa học, tác nhân gây bệnh (nấm, noãn khuẩn, virus).
* Triệu chứng trực quan trên lá cây.
* Điều kiện môi trường thuận lợi (nhiệt độ, ẩm độ).
* Phác đồ hóa học (hoạt chất, tên thương mại ở VN, liều lượng, thời gian cách ly - PHI).
* Biện pháp sinh học & hữu cơ (Trichoderma, Bacillus subtilis,...).
* Biện pháp canh tác phòng ngừa.
* Cảnh báo an toàn & bảo hộ lao động (PPE).


In [ ]:
# Khởi tạo engine tri thức nông nghiệp
advisory_engine = AdvisoryKnowledgeBase(KNOWLEDGE_BASE_PATH)

# Lấy danh mục tất cả các bệnh và hiển thị dưới dạng bảng DataFrame
diseases_list = advisory_engine.list_all_diseases()
df_diseases = pd.DataFrame(diseases_list)
print("DANH MỤC TẤT CẢ CÁC BỆNH TRÊN CÀ CHUA, KHOAI TÂY, NGÔ:")
print(df_diseases[["disease_id", "crop", "vietnamese_name", "pathogen_type"]].to_string())

# Tra cứu chi tiết một bệnh điển hình (Ví dụ: Bệnh đốm vòng cà chua - Tomato Early Blight)
sample_disease = advisory_engine.get_disease_by_id("Tomato_Early_blight")
print("\n" + "="*80)
print(f"CHI TIẾT PHÁC ĐỒ ĐIỀU TRỊ: {sample_disease['vietnamese_name'].upper()}")
print("="*80)
print(f"- Tác nhân gây bệnh: {sample_disease['pathogen_name']} ({sample_disease['pathogen_type']})")
print(f"- Triệu chứng: {sample_disease['visual_symptoms']}")
print(f"- Điều kiện bùng phát: {sample_disease['environmental_triggers']}")
print("\n* Thuốc BVTV khuyến cáo:")
for idx, chem in enumerate(sample_disease['chemical_controls'], 1):
    print(f"  {idx}. Hoạt chất: {chem['active_ingredient']} (Biệt dược: {', '.join(chem['trade_names'])})")
    print(f"     + Liều lượng: {chem['dosage']} | Cách ly (PHI): {chem['pre_harvest_interval_days']} ngày")
    print(f"     + Hướng dẫn: {chem['application_notes']}")

print("\n* Biện pháp canh tác phòng ngừa:")
for idx, cult in enumerate(sample_disease['cultural_preventions'], 1):
    print(f"  {idx}. {cult}")

print(f"\n* Cảnh báo an toàn (PPE): {sample_disease['safety_precautions'][0]}")


---
## 📸 Module 2: Pipeline Tiền xử lý & Input Validation (Subsystem A)

Pipeline tiếp nhận ảnh tải lên từ camera hoặc file upload của người dùng:
1. **Validation:** Kiểm tra dung lượng file, định dạng tệp (JPEG/PNG), phát hiện và loại bỏ các file rác, file lỗi hoặc file bị hỏng.
2. **Preprocessing:** Chuyển đổi không gian màu RGB, resize chuẩn $224 \times 224$ pixels, chuẩn hóa Z-score với bộ tham số chuẩn ImageNet (Mean: `[0.485, 0.456, 0.406]`, Std: `[0.229, 0.224, 0.225]`).


In [ ]:
# Kiểm tra tính năng lọc file lỗi và bảo vệ hệ thống
corrupt_path = SAMPLE_IMAGES_DIR / "corrupted_image.jpg"
valid_path = SAMPLE_IMAGES_DIR / "tomato_leaf_sample.jpg"

print("--- KIỂM TRA BỘ LỌC FILE HỎNG (VALIDATION) ---")
try:
    with open(corrupt_path, "rb") as f:
        validate_image_file(f.read(), "corrupted_image.jpg")
except InputValidationError as e:
    print(f"[OK] Phát hiện và chặn file lỗi thành công: {e}")

# Tiền xử lý ảnh hợp lệ
with open(valid_path, "rb") as f:
    valid_bytes = f.read()

validate_image_file(valid_bytes, "tomato_leaf_sample.jpg")
tensor_input = preprocess_image(valid_bytes)

print(f"\n[OK] Tiền xử lý ảnh thành công!")
print(f"Tensor Shape: {tensor_input.shape} (Batch, Channel, Height, Width)")
print(f"Tensor Min: {tensor_input.min():.2f}, Max: {tensor_input.max():.2f}")

# Hiển thị trực quan ảnh gốc và kích thước resize 224x224
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
raw_img = Image.open(valid_path)
axes[0].imshow(raw_img)
axes[0].set_title(f"Ảnh gốc ({raw_img.size[0]}x{raw_img.size[1]})")
axes[0].axis("off")

# Unnormalize tensor để hiển thị ảnh đầu vào mạng neural
unnorm_tensor = tensor_input[0].clone().permute(1, 2, 0).numpy()
unnorm_tensor = unnorm_tensor * IMAGENET_STD + IMAGENET_MEAN
unnorm_tensor = np.clip(unnorm_tensor, 0, 1)

axes[1].imshow(unnorm_tensor)
axes[1].set_title("Ảnh đầu vào MobileNetV2 (224x224 RGB)")
axes[1].axis("off")
plt.tight_layout()
plt.show()


---
## ⚡ Module 3: MobileNetV2 Inference Engine & Đo lường Latency (< 200ms)

Theo chỉ tiêu kỹ thuật của đề tài Capstone:
* **Mô hình học sâu:** CNN MobileNetV2 tối ưu cho thiết bị biên và hệ thống web nhẹ.
* **Thời gian đáp ứng (Response Latency):** Phải đạt trung bình **dưới 200 milliseconds** trên một lượt suy luận.


In [ ]:
# Khởi tạo Inference Engine
engine = CropDiseaseInferenceEngine()

# Chạy suy luận trên ảnh mẫu và đo độ trễ
result = engine.predict(valid_bytes)

print("="*60)
print("KẾT QUẢ INFERENCE MÔ HÌNH MOBILENETV2")
print("="*60)
print(f"Cây trồng nhận diện: {result['crop']}")
print(f"Dự đoán nhãn bệnh: {result['predicted_class']}")
print(f"Độ tin cậy (Confidence): {result['confidence_percentage']}")
print(f"Thời gian xử lý (Latency): {result['latency_ms']} ms")
print(f"Đạt tiêu chuẩn Capstone (< 200ms): {'ĐẠT CHUẨN' if result['benchmark_met'] else 'KHÔNG ĐẠT'}")

# Trực quan hóa phân phối xác suất dự đoán (Top Classes)
probs = result['all_probabilities']
sorted_probs = dict(sorted(probs.items(), key=lambda item: item[1], reverse=True))

plt.figure(figsize=(10, 5))
classes = list(sorted_probs.keys())[:5]
values = [sorted_probs[c] for c in classes]
bars = plt.barh(classes[::-1], values[::-1], color="#2E7D32")
plt.xlabel("Xác suất dự đoán (Softmax Probability)")
plt.title("Top 5 Dự đoán của Mô hình MobileNetV2")
plt.xlim(0, max(max(values) * 1.2, 0.2))
for bar in bars:
    w = bar.get_width()
    plt.text(w + 0.005, bar.get_y() + bar.get_height()/2, f"{w*100:.1f}%", va='center')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()


---
## 🛡️ Module 4: Bộ điều phối Logic Quyết định (Confidence-Gated Decision Engine)

Để bảo vệ người nông dân tránh việc phun nhầm thuốc hóa học khi ảnh chụp mờ hoặc mô hình không chắc chắn:
* **Nếu $\text{Confidence} \ge \text{Threshold}$ (75%):** Hệ thống xác nhận bệnh (`CONFIRMED`), trích xuất toàn bộ phác đồ trị bệnh, hoạt chất, cách ly và quy định bảo hộ.
* **Nếu $\text{Confidence} < \text{Threshold}$:** Kích hoạt cơ chế an toàn (`UNCERTAIN`), tạm thời không xuất đơn thuốc hóa học, gửi hướng dẫn yêu cầu chụp lại ảnh rõ hơn hoặc liên hệ cán bộ khuyến nông địa phương.


In [ ]:
# Kiểm tra 2 kịch bản của Bộ lọc Confidence Gate:

print("--- KỊCH BẢN 1: ĐỘ TIN CẬY CAO (>= 75%) ---")
mock_high_conf = {
    "predicted_class": "Tomato_Early_blight",
    "crop": "Tomato",
    "is_healthy": False,
    "confidence": 0.924,
    "latency_ms": 110.5
}
payload_high = advisory_engine.process_diagnosis(mock_high_conf)
print(f"Trạng thái: {payload_high['diagnosis_status']}")
print(f"Thông báo: {payload_high['message']}")
print(f"Tên bệnh: {payload_high['advisory_details']['vietnamese_name']}")
print(f"Số lượng thuốc BVTV khuyến cáo: {len(payload_high['advisory_details']['chemical_controls'])} loại")

print("\n--- KỊCH BẢN 2: ĐỘ TIN CẬY THẤP (< 75%) - BẢO VỆ AN TOÀN ---")
mock_low_conf = {
    "predicted_class": "Tomato_Early_blight",
    "crop": "Tomato",
    "is_healthy": False,
    "confidence": 0.582,
    "latency_ms": 105.2
}
payload_low = advisory_engine.process_diagnosis(mock_low_conf)
print(f"Trạng thái: {payload_low['diagnosis_status']}")
print(f"Cảnh báo: {payload_low['remediation_notice']}")
print(f"Hướng dẫn người dân: {payload_low['advisory_details']['guidance_for_farmer'][0]}")


---
## 🔌 Module 5 & 6: RESTful API với FastAPI & Kiểm thử Toàn diện (End-to-End Test Suite)

Dưới đây là việc khởi tạo và kiểm thử toàn bộ hệ sinh thái API của bạn Khánh bằng `TestClient` ngay bên trong Notebook!


In [ ]:
# Khởi tạo client kiểm thử FastAPI
client = TestClient(app)

print("--- 1. TEST ENDPOINT /api/health ---")
res_health = client.get("/api/health")
print(f"Status Code: {res_health.status_code}")
print(json.dumps(res_health.json(), indent=2, ensure_ascii=False))

print("\n--- 2. TEST ENDPOINT /api/crops ---")
res_crops = client.get("/api/crops")
print(json.dumps(res_crops.json(), indent=2, ensure_ascii=False))

print("\n--- 3. TEST ENDPOINT /api/diagnose (UPLOAD ẢNH LÁ CÂY) ---")
with open(valid_path, "rb") as f:
    res_diag = client.post(
        "/api/diagnose",
        files={"file": ("tomato_leaf_sample.jpg", f, "image/jpeg")}
    )
print(f"Status Code: {res_diag.status_code}")
diag_data = res_diag.json()
print(f"Status chẩn đoán: {diag_data['diagnosis_status']}")
print(f"Độ trễ xử lý: {diag_data['prediction']['latency_ms']} ms")
print(f"Độ tin cậy: {diag_data['prediction']['confidence_percentage']}")
print(f"Quy định pháp lý & PPE: {diag_data['regulatory_disclaimer'][:80]}...")


---
## 🚀 Hướng dẫn Triển khai & Tích hợp với Frontend (Ma Hồng Hải)

Khi bạn muốn chạy server backend độc lập cho cả nhóm kết nối kiểm thử giao diện:

1. Mở Terminal mới trong VS Code.
2. Di chuyển vào thư mục dự án và chạy lệnh:
   ```bash
   uvicorn backend.main:app --host 0.0.0.0 --port 8000 --reload
   ```
3. Các đường link quan trọng:
   * **Trang API Documentation tương tác (Swagger UI):** `http://localhost:8000/docs`
   * **Endpoint chẩn đoán bệnh:** `http://localhost:8000/api/diagnose` [POST]
   * **Kiểm tra trạng thái server:** `http://localhost:8000/api/health` [GET]

🎉 **Phần Backend của Nguyễn Quốc Khánh đã hoàn thành xuất sắc và sẵn sàng tích hợp!**
